In [ ]:
# import all libraries
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
import numpy as np
import cv2
import argparse
import os

block_size = 64 # 32, 64, 128, 256
reduce_size = 64
# threshold = 4
clean_path = "data/clean"
data_path = "data/defocus"
# data_path = "data/mobile"

In [ ]:
# Define fft functions
def fft1d(x, n, model):
    if len(x) != n:
        raise ValueError(f'Input must be length {n}')
    input_vec = x.real[np.newaxis, :]
    pred = model.predict(input_vec, verbose=0)[0]
    result = pred[:n] + 1j * np.zeros_like(pred)
    return result

def fft2d(image, n, model):
    if image.shape != (n, n):
        raise ValueError(f'Input image must be shape ({n}, {n})')
    row = np.array([fft1d(row, n, model) for row in image]) # row
    result = np.array([fft1d(col, n, model) for col in row.T]).T # col
    return result



In [ ]:
# Define deteciton function
def detect_blur_fft(image, n, k, size=60, threshold=10, vis=False):

    model_path = os.path.join(os.getcwd(), 'model', f'1D_FFT_{n}_reduce_{k}.h5')
    model_1d = load_model(model_path)

    (h, w) = image.shape
    (cx, cy) = (int(w / 2.0), int(h / 2.0))
    fft = fft2d(image, n, model_1d)
    fft_shift = np.fft.fftshift(fft)

    if vis:
        magnitude = 20 * np.log(np.abs(fft_shift))

        (fig, ax) = plt.subplots(1, 2,)

        ax[0].imshow(image, cmap="gray")
        ax[0].set_title("Input")
        ax[0].set_xticks([])
        ax[0].set_yticks([])

        ax[1].imshow(magnitude, cmap="gray")
        ax[1].set_title("Magnitude Spectrum")
        ax[1].set_xticks([])
        ax[1].set_yticks([])

        plt.show()

    fft_shift[cy - size : cy + size, cx - size : cx + size] = 0
    fft_shift = np.fft.ifftshift(fft_shift)
    recon = np.fft.ifft2(fft_shift)

    magnitude = 20 * np.log(np.abs(recon))
    mean = np.mean(magnitude)

    return mean, mean <= threshold

In [ ]:
# Define threshold function
def detect_threshold(image, n, k, size=60):

    model_path = os.path.join(os.getcwd(), 'model', f'1D_FFT_{n}_reduce_{k}.h5')
    model_1d = load_model(model_path)

    (h, w) = image.shape
    (cx, cy) = (int(w / 2.0), int(h / 2.0))
    fft = fft2d(image, n, model_1d)
    fft_shift = np.fft.fftshift(fft)

    fft_shift[cy - size : cy + size, cx - size : cx + size] = 0
    fft_shift = np.fft.ifftshift(fft_shift)
    recon = np.fft.ifft2(fft_shift)

    magnitude = 20 * np.log(np.abs(recon))
    mean = np.mean(magnitude)

    return mean

In [ ]:
# initiate recording files
results_file = 'outtxt/results_'+str(block_size)+'_reduce_'+str(reduce_size)+'.txt'
with open(results_file, "a") as f:
    f.write("Image_Name, Mean, Blurry_Status\n")
    
# Detect threshold
threshold = 0
for root, sub_dirs, files in os.walk(clean_path):
    for file in files:

        print("\nDetect "+file+" using "+f'1D_FFT_{block_size}_reduce_{reduce_size}.h5')
        image = cv2.imread(os.path.join(root, file))
        h, w, r = image.shape
        gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        gray_img = gray_img[128:128+1024, 256:256+1536] # cropped image size
        gray = cv2.resize(gray_img, (0,0), fx=0.5, fy=0.5) 

        metrics = []
        for y in range(0, h, block_size):
            for x in range(0, w, block_size):
                block = gray[y:min(y + block_size, h), x:min(x + block_size, w)]
                if block.shape[0] == block_size and block.shape[1] == block_size:
                    mean= detect_threshold(block, block_size, reduce_size, size=60)
                    metrics.append(mean)
        threshold = np.mean(metrics)

        res = np.dstack([gray] * 3)
        color = (0, 255, 0)
        text = "Threshold ({:.4f})"
        text = text.format(threshold)
        cv2.putText(res, text, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        print("Info: ", text, " ", file)
        cv2.imwrite("outimg/Det_"+f'FFT_{block_size}_reduce_{reduce_size}_'+file, res)
        print("Finish "+file+" using "+f'1D_FFT_{block_size}_reduce_{reduce_size}.h5'+'\n')

# Detect blury
for root, sub_dirs, files in os.walk(data_path):
    for file in files:

        print("\nDetect "+file+" using "+f'1D_FFT_{block_size}_reduce_{reduce_size}.h5')
        image = cv2.imread(os.path.join(root, file))
        h, w, r = image.shape
        gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        gray_img = gray_img[128:128+1024, 256:256+1536] # cropped image size
        gray = cv2.resize(gray_img, (0,0), fx=0.5, fy=0.5) 

        metrics = []
        for y in range(0, h, block_size):
            for x in range(0, w, block_size):
                block = gray[y:min(y + block_size, h), x:min(x + block_size, w)]
                if block.shape[0] == block_size and block.shape[1] == block_size:
                    mean, _ = detect_blur_fft(block, block_size, reduce_size, size=60, threshold=20, vis=False)
                    metrics.append(mean)
        result = np.mean(metrics)
        blurry = result < threshold

        res = np.dstack([gray] * 3)
        color = (0, 0, 255) if blurry else (0, 255, 0)
        text = "Blurry ({:.4f})" if blurry else "Not Blurry ({:.4f})"
        text = text.format(result)
        cv2.putText(res, text, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        print("Info: ", text, " ", file)

        # cv2.imshow("Output", res)
        # cv2.waitKey(0)
        cv2.imwrite("outimg/Det_"+f'FFT_{block_size}_reduce_{reduce_size}_'+file, res)

        with open(results_file, "a") as f:
            f.write("{}, {:.4f}, {}\n".format(file, result, "Blurry" if blurry else "Not Blurry"))
            
        print("Finish "+file+" using "+f'1D_FFT_{block_size}_reduce_{reduce_size}.h5'+'\n')